In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,NotRequired
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage
from dotenv import load_dotenv
from pydantic import BaseModel,Field

In [107]:
load_dotenv()
generator_llm=ChatOpenAI(model='gpt-4o-mini')
evaluator_llm=ChatOpenAI(model='gpt-4o-mini')
optimizer_llm=ChatOpenAI(model='gpt-4o-mini')

In [ ]:
class PostState(TypedDict):
    topic: str
    platform: str
    goal: str
    audience: str
    tone: str
    key_points: str
    evaluation: Literal['approved', 'needs_improvement']
    iteration: int
    max_iteration: int
    post: NotRequired[str]
    feedback: NotRequired[str]


In [109]:
def generate_post(state:PostState):

    # prompt
    messages = [
        SystemMessage(
            content="""
    You are an expert social media content writer who creates engaging,
    original, and platform-specific posts.
    """
        ),
        HumanMessage(
            content=f"""
    Write a high-quality social media post for {state['platform']} on the topic:
    "{state['topic']}".

    Target audience: {state['audience']}
    Goal: {state['goal']}
    Tone: {state['tone']}

    Key points to include:
    {state['key_points']}

    Rules:
    - Start with a strong hook that makes people stop scrolling.
    - Keep the content natural, simple, and human-like.
    - Use short paragraphs and clean formatting for readability.
    - Include a practical insight, relatable example, or strong opinion.
    - Do not use question-answer format.
    - Avoid generic AI phrases, repetitive wording, and unnecessary emojis.
    - Add a clear call to action at the end.
    - Add 5–8 relevant hashtags if the platform is LinkedIn or Instagram.
    - Keep it within the ideal character limit for {state['platform']}.
    - This is version {state['iteration'] + 1}.
    - Return only the final post. Do not add explanations, titles, or notes.
    """
        )
    ]
    
    response = generator_llm.invoke(messages).content
    return{
        'post':response
    }


In [110]:
class PostEvaluateSchema(BaseModel):
    evaluation:Literal['approved','needs_improvement']=Field(description="final evaluation result")
    feedback:str=Field(description="Constructive feedback for the post")
    

In [111]:
def evaluate_post(state:PostState):
    # evaluator_prompt
    messages = [
        SystemMessage(
            content="""
    You are a strict, practical social media content reviewer.
    You evaluate posts based on platform fit, clarity, engagement potential,
    originality, and whether they follow the requested requirements.
    """
        ),
        HumanMessage(
            content=f"""
    Evaluate the following social media post.

    Platform: {state['platform']}
    Topic: {state['topic']}
    Target audience: {state['audience']}
    Goal: {state['goal']}
    Requested tone: {state['tone']}

    Post:
    \"\"\"{state['post']}\"\"\"

    Evaluate it using these criteria:

    1. Hook — Does the first line grab attention and encourage people to keep reading?
    2. Platform Fit — Is the format, length, tone, and structure suitable for {state['platform']}?
    3. Audience Relevance — Does it speak clearly to {state['audience']}?
    4. Goal Alignment — Does it support the goal: {state['goal']}?
    5. Clarity & Readability — Is it simple, easy to scan, and written in short, clear paragraphs?
    6. Value — Does it provide a useful insight, relatable example, opinion, or takeaway?
    7. Originality — Does it avoid generic, overused, or obvious AI-style wording?
    8. Tone — Does it match the requested tone: {state['tone']}?
    9. Call to Action — Does it end with a clear and relevant CTA?
    10. Hashtags — If the platform is LinkedIn or Instagram, are there 5–8 relevant and non-spammy hashtags?
    11. Rule Compliance — Check whether it:
    - Uses unnecessary emojis
    - Uses repetitive wording
    - Includes generic AI phrases
    - Contains irrelevant information
    - Exceeds the ideal length for {state['platform']}
    - Adds explanations, titles, or notes instead of only the post

    Auto-reject if:
    - The post does not match the topic.
    - It does not match the target audience or goal.
    - It has no meaningful hook.
    - It is generic, overly promotional, or clearly AI-generated.
    - It has no useful value or takeaway.
    - It does not include a CTA.
    - It violates important platform-specific requirements.

    Respond ONLY in this structured format:

    evaluation: "approved" or "needs_improvement"
    feedback: "<specific actionable improvements>"
    """
        )
    ]
    
    structure_model_llm=evaluator_llm.with_structured_output(PostEvaluateSchema)
    response = structure_model_llm.invoke(messages)
    return{
        'evaluation':response.evaluation,
        'feedback':response.feedback,
        
    }

In [112]:
def optimize_post(state:PostState):
        # improvement_prompt
    messages = [
        SystemMessage(
            content="""
    You are an expert social media content editor.
    You improve posts using evaluator feedback while preserving the original
    topic, audience, goal, and requested tone.
    """
        ),
        HumanMessage(
            content=f"""
    Improve the social media post below using the evaluator feedback.

    Platform: {state['platform']}
    Topic: {state['topic']}
    Target audience: {state['audience']}
    Goal: {state['goal']}
    Requested tone: {state['tone']}

    Original post:
    \"\"\"{state['post']}\"\"\"

    Evaluator feedback:
    \"\"\"{state['feedback']}\"\"\"

    Rewrite requirements:
    - Fix every meaningful issue mentioned in the feedback.
    - Keep the post focused on the original topic: {state['topic']}.
    - Make the first line stronger and more scroll-stopping.
    - Make it relevant to {state['audience']}.
    - Support the goal: {state['goal']}.
    - Maintain this tone: {state['tone']}.
    - Use short paragraphs and simple, natural language.
    - Add a practical insight, example, opinion, or takeaway if missing.
    - Remove generic AI-style phrases, repetitive wording, and unnecessary emojis.
    - End with a clear, relevant call to action.
    - If the platform is LinkedIn or Instagram, include 5–8 relevant hashtags.
    - Keep it within the ideal character limit for {state['platform']}.
    - This is improved version {state['iteration'] + 1}.

    Return ONLY the rewritten post.
    Do not include explanations, labels, notes, or quotation marks.
    """
        )
    ]
    
    response= optimizer_llm.invoke(messages).content
    iteration = state['iteration']+1
    return{
        'post':response,
        'iteration':iteration
    }

In [113]:
def route_evaluation(state:PostState):
    if state['evaluation'] =='approved' or state['iteration']>=state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [114]:
graph = StateGraph(PostState)

graph.add_node('generate',generate_post)
graph.add_node('evaluate',evaluate_post)
graph.add_node('optimize',optimize_post)


graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')

graph.add_conditional_edges('evaluate',route_evaluation,{'approved':END,'needs_improvement':'optimize'})

graph.add_edge('optimize','evaluate')

workflow = graph.compile()

In [115]:
initial_state = {
    "topic": "Why React developers should learn Next.js",
    "platform": "LinkedIn",
    "goal": "Build personal brand and increase engagement from frontend developers.",
    "audience": "Frontend developers with 1 to 5 years of experience who already know React.js.",
    "tone": "Professional, practical, friendly, and confident.",
    "key_points": """
- React is great for building user interfaces.
- Next.js adds SEO, routing, server-side rendering, API routes, and performance optimization.
- Next.js helps developers build production-ready applications.
- React developers can learn Next.js quickly because it is built on React.
- Examples include blogs, e-commerce websites, SaaS dashboards, and portfolios.
""".strip(),
    "evaluation": "need_improvements",
    "iteration": 0,
    "max_iteration": 3
}

result = workflow.invoke(initial_state)
print(result)

{'topic': 'Why React developers should learn Next.js', 'post': "Ready to elevate your React skills? Learning Next.js could be the game-changer you didn’t know you needed.\n\nWhile React is fantastic for creating dynamic user interfaces, Next.js takes your development experience to the next level by adding essential features like SEO capabilities, routing, server-side rendering, and performance optimization. Imagine transforming a standard React app into a lightning-fast, production-ready application with built-in API routes!\n\nNext.js helps you efficiently build diverse projects—from blogs and e-commerce sites to SaaS dashboards and personal portfolios. If you’re already comfortable with React, picking up Next.js will feel like a natural progression. The enhanced capabilities ensure you’re not just coding but creating user experiences that stand out in today’s competitive landscape.\n\nDon't miss out on the opportunity to broaden your skill set and increase your marketability as a fro